# Digital Twin with RAG — Christopher Burt

**Community contribution** extending Labs 3–4.

```
cb_digital_twin/
├── CB_Digital_Twin_RAG.ipynb   ← this notebook
├── cb_me/
│   ├── summary.txt             ← your bio
│   └── linkedin.pdf            ← your LinkedIn export
└── rag_data/                   ← vector DB (auto-generated, gitignored)
```

### Problem
Labs 3–4 inject a static LinkedIn PDF and `summary.txt` into the system prompt. That misses skills listed on my website ([christopherburt.dev](https://www.christopherburt.dev)) and does not scale as content grows.

### Solution
A **Retrieval-Augmented Generation (RAG)** chatbot that indexes three knowledge sources — personal summary, LinkedIn PDF, and live website content — then retrieves only the most relevant chunks per question.

### Stack
- **OpenAI** — `gpt-4o-mini` for chat, `text-embedding-3-small` for embeddings
- **ChromaDB** — local vector store (`rag_data/`)
- **Gradio** — chat UI
- **Lab 4 tools** — Pushover notifications for contact capture and unknown questions

### Personal data
All project files live in `community_contributions/cb_digital_twin/`. Personal data is in `cb_me/` (not the course `me/` folder) so lab templates stay untouched.

**Author:** Christopher Burt

In [1]:
from pathlib import Path

# YOUR files live here — keeps course me/ (Ed's template) untouched for labs
CB_ME_DIR = Path("./cb_me").resolve()
if not (CB_ME_DIR / "summary.txt").exists():
    raise FileNotFoundError(
        f"Missing {CB_ME_DIR / 'summary.txt'}. "
        "Your personal files belong in cb_digital_twin/cb_me/"
    )

LINKEDIN_PDF = CB_ME_DIR / "linkedin.pdf"
if not LINKEDIN_PDF.exists():
    raise FileNotFoundError(
        f"Missing {LINKEDIN_PDF}.\n"
        "Download YOUR LinkedIn as PDF and save it to cb_digital_twin/cb_me/linkedin.pdf"
    )

print("Personal data folder:", CB_ME_DIR)
print("LinkedIn PDF:", LINKEDIN_PDF.name)

documents = {
    "summary": None,
    "linkedin": None,
    "website": None,
}

Personal data folder: /Users/cburto/projects/agents/1_foundations/community_contributions/cb_me
LinkedIn PDF: linkedin.pdf


In [2]:
# Summary (your file — not the course me/summary.txt)
with open(CB_ME_DIR / "summary.txt", encoding="utf-8") as f:
    documents["summary"] = f.read()

In [3]:
from pypdf import PdfReader

reader = PdfReader(str(LINKEDIN_PDF))
documents["linkedin"] = "".join(
    page.extract_text() or "" for page in reader.pages
)

In [4]:
import requests
from bs4 import BeautifulSoup

def fetch_website_text(url):
    r = requests.get(url, timeout=15, headers={"User-Agent": "CBDigitalTwin/1.0"})
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer"]):
        tag.decompose()
    lines = [ln.strip() for ln in soup.get_text("\n").splitlines() if ln.strip()]
    return "\n".join(lines)

documents["website"] = fetch_website_text("https://www.christopherburt.dev")

In [5]:
# Guardrail: catch course template files before we embed the wrong person into the vector DB
OWNER_NAME = "Christopher Burt"
TEMPLATE_MARKERS = ["Ed Donner", "eddonner", "ed.donner@gmail.com", "Nebula.io"]

def validate_owner_text(label, text):
    for marker in TEMPLATE_MARKERS:
        if marker.lower() in text.lower():
            raise ValueError(
                f"{label} still looks like the course template (found '{marker}').\n"
                f"Use YOUR files in cb_digital_twin/cb_me/ — not 1_foundations/me/."
            )
    if OWNER_NAME not in text:
        print(f"Warning: '{OWNER_NAME}' not found in {label} — double-check cb_me/ files.")

validate_owner_text("summary", documents["summary"])
validate_owner_text("linkedin", documents["linkedin"])
print(f"Owner check passed. LinkedIn loaded ({len(documents['linkedin']):,} chars).")

Owner check passed. LinkedIn loaded (4,261 chars).


In [6]:
print(documents["website"][:800])   # first 800 characters

Christopher Burt — AI Automation Engineer
CB
Christopher Burt
AI Automation Engineer
Building intelligent systems
that
think, act, and scale.
I'm Christopher Burt — an AI Automation Engineer focused on the practical work of
bringing AI into the enterprise. From agentic systems and retrieval-augmented
chatbots to rollouts, governance, and the engineering glue that keeps it all
reliable in production.
See what I do
Get in touch →
Working with
OpenAI
Anthropic
LangChain
LlamaIndex
Pinecone
Glean
LangSmith
n8n
ElevenLabs
About / Capabilities
A toolkit for the
AI-native enterprise.
I work across the full stack of modern AI delivery — from foundation models and
agent frameworks to enterprise rollout, governance, and the unglamorous engineering
that keeps everything running in production.
01
Core


In [7]:
for name, text in documents.items():
    print(f"{name}: {len(text):,} characters")

summary: 479 characters
linkedin: 4,261 characters
website: 2,918 characters


## Step 2 — Chunking

**Why we chunk:** Your three documents are one long string each. Retrieval works best on *focused* passages — e.g. a chunk about vector databases, not your entire LinkedIn PDF.

**How it works:** We split text into overlapping windows of ~400 words. Overlap prevents sentences from being cut in half at chunk boundaries.

**Output:** A list called `all_chunks` — each item has `id`, `text`, and `source` (so we know if an answer came from website vs LinkedIn).

In [8]:
def chunk_text(text, chunk_size=400, overlap=50):
    """Split text into overlapping word windows."""
    words = text.split()
    chunks = []
    step = chunk_size - overlap  # how far we advance each window
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# Quick demo on a tiny string so you can see overlap in action
demo = chunk_text("one two three four five six seven eight nine ten", chunk_size=4, overlap=2)
for i, c in enumerate(demo):
    print(f"chunk {i}: {c}")

chunk 0: one two three four
chunk 1: three four five six
chunk 2: five six seven eight
chunk 3: seven eight nine ten
chunk 4: nine ten


In [9]:
all_chunks = []
for source, content in documents.items():
    for i, chunk in enumerate(chunk_text(content)):
        all_chunks.append({
            "id": f"{source}_{i}",
            "text": chunk,
            "source": source,
        })

print(f"Created {len(all_chunks)} chunks from {len(documents)} documents")
for source in documents:
    count = sum(1 for c in all_chunks if c["source"] == source)
    print(f"  {source}: {count} chunks")

Created 5 chunks from 3 documents
  summary: 1 chunks
  linkedin: 2 chunks
  website: 2 chunks


In [10]:
# Checkpoint: find a website chunk that mentions vector databases
for c in all_chunks:
    if c["source"] == "website" and "vector" in c["text"].lower():
        print(f"[{c['id']}] {c['text'][:200]}...")
        break

[website_0] Christopher Burt — AI Automation Engineer CB Christopher Burt AI Automation Engineer Building intelligent systems that think, act, and scale. I'm Christopher Burt — an AI Automation Engineer focused o...


### Rebuilding the vector database

**Re-run cells 0 → 12** whenever you change:
- `cb_me/summary.txt`
- `cb_me/linkedin.pdf`
- your website content (cell 3 re-scrapes on each run)

Cell 12 **deletes the old collection and re-embeds everything** — you do not need to manually delete `rag_data/`.

After rebuilding, re-run cells 14–15 to test retrieval, then 17–19 for chat.

## Step 3 — Embeddings + ChromaDB

**Why embeddings?** Computers can't search by *meaning* with plain text matching alone. Embeddings turn each chunk into a list of numbers (a vector) that captures semantic similarity — so a question about "vector DBs" lands near chunks mentioning Pinecone and Milvus.

**Why ChromaDB?** A local vector database that stores chunks + their embeddings and lets us run similarity search. Data is saved to `./rag_data/` next to this notebook.

**Cost note:** We use OpenAI's `text-embedding-3-small` — a few cents for ~5 chunks.

In [11]:
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv(override=True)
openai = OpenAI()

def embed_texts(texts):
    """Convert a list of strings into embedding vectors via OpenAI."""
    response = openai.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    return [item.embedding for item in response.data]

In [12]:
from pathlib import Path

# Named folder so it's easy to find next to this notebook in community_contributions/
RAG_DIR = Path("./rag_data")
RAG_DIR.mkdir(exist_ok=True)
print("Vector DB path:", RAG_DIR.resolve())

client = chromadb.PersistentClient(path=str(RAG_DIR))

# Fresh index each run (fine for a small personal knowledge base)
try:
    client.delete_collection("christopher_burt")
except Exception:
    pass

collection = client.create_collection("christopher_burt")

batch_size = 50
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i : i + batch_size]
    embeddings = embed_texts([c["text"] for c in batch])
    collection.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        embeddings=embeddings,
        metadatas=[{"source": c["source"]} for c in batch],
    )

print("Vector store ready:", collection.count(), "chunks")

Vector DB path: /Users/cburto/projects/agents/1_foundations/community_contributions/CB_Digital_Twin_rag_data
Vector store ready: 5 chunks


## Step 4 — Retrieval

**What happens here:** When a user asks a question, we:
1. Embed the question (same way we embedded chunks)
2. Ask ChromaDB: "which stored chunks are closest to this question?"
3. Return the top matches to inject into the prompt

This is the **R** in RAG — **Retrieval**.

In [13]:
def retrieve(query, top_k=3):
    """Find the chunks most semantically similar to the user's question."""
    query_embedding = embed_texts([query])[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "text": doc,
            "source": meta["source"],
            "distance": dist,  # lower = more similar
        })
    return retrieved

In [14]:
# Checkpoint: this is the question that failed in Lab 4 without website content
question = "Do you have experience with vector databases?"

for i, r in enumerate(retrieve(question), start=1):
    print(f"\n--- Match {i} [{r['source']}] distance={r['distance']:.4f} ---")
    print(r["text"][:300], "...")


--- Match 1 [website] distance=1.4295 ---
Christopher Burt — AI Automation Engineer CB Christopher Burt AI Automation Engineer Building intelligent systems that think, act, and scale. I'm Christopher Burt — an AI Automation Engineer focused on the practical work of bringing AI into the enterprise. From agentic systems and retrieval-augmente ...

--- Match 2 [website] distance=1.5067 ---
workflows, model and agent monitoring with tools like LangSmith, and deploying solutions through automated pipelines. Contact Let's connect! I'm always open to interesting AI conversations — happy to chat about the space, swap notes, or stay in touch. Email christopher.r.burt@gmail.com Based Remote  ...

--- Match 3 [summary] distance=1.5157 ---
My name is Christopher Burt. I currently work for the illustrious firm Ping Identity where I work as an AI Automation Engineer. I am based in McLean, Virginia with a loving wife and two beautiful girls. I LOVE to travel! I have been to every continent in the wo

## Step 5 — RAG + Chat + Tools

**The full loop (this is the complete RAG chatbot):**

1. User sends a message
2. `retrieve()` finds relevant chunks
3. Chunks go into the system prompt (the **A** in RAG — **Augmented** generation)
4. GPT answers in character, with Lab 4 tools (Pushover notifications)

In [15]:
import json
import os
import requests
import gradio as gr

name = "Christopher Burt"

# Pushover (from Lab 4) — optional; chat still works if keys are missing
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(message):
    print(f"Push: {message}")
    if pushover_user and pushover_token:
        requests.post(pushover_url, data={
            "user": pushover_user, "token": pushover_token, "message": message
        })

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

tools = [
    {"type": "function", "function": {
        "name": "record_user_details",
        "description": "Use when a user wants to get in touch and provided an email address",
        "parameters": {
            "type": "object",
            "properties": {
                "email": {"type": "string", "description": "The user's email"},
                "name": {"type": "string", "description": "The user's name, if provided"},
                "notes": {"type": "string", "description": "Brief context from the conversation"},
            },
            "required": ["email"],
            "additionalProperties": False,
        },
    }},
    {"type": "function", "function": {
        "name": "record_unknown_question",
        "description": "Always use when you cannot answer a question from the retrieved context",
        "parameters": {
            "type": "object",
            "properties": {
                "question": {"type": "string", "description": "The unanswered question"},
            },
            "required": ["question"],
            "additionalProperties": False,
        },
    }},
]

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id,
        })
    return results

In [16]:
def build_system_prompt(rag_context):
  prompt = f"""You are acting as {name}. You are answering questions on {name}'s website,
particularly about career, background, skills, and experience.
Stay in character — professional and engaging, as if speaking to a potential client or employer.

Answer using ONLY the retrieved context below. If the context does not contain the answer,
use your record_unknown_question tool.

If the user is interested in connecting, ask for their email and use record_user_details.

## Retrieved context:
"""
  for item in rag_context:
      prompt += f"\n[{item['source']}]:\n{item['text']}\n"
  prompt += f"\nWith this context, chat with the user as {name}."
  return prompt

def chat(message, history):
    rag_context = retrieve(message, top_k=3)
    messages = (
        [{"role": "system", "content": build_system_prompt(rag_context)}]
        + history
        + [{"role": "user", "content": message}]
    )
    done = False
    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini", messages=messages, tools=tools
        )
        if response.choices[0].finish_reason == "tool_calls":
            reply = response.choices[0].message
            messages.append(reply)
            messages.extend(handle_tool_calls(reply.tool_calls))
        else:
            done = True
    return response.choices[0].message.content

In [17]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
